In [1]:
#!/usr/bin/env python3.12.7
# -*- coding: utf-8 -*-
"""
Created on Tue Dec 19 15:25:43 2023
@author: marriyapillais
"""

'\nCreated on Tue Dec 19 15:25:43 2023\n@author: marriyapillais\n'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Reading MSW data downloaded from Eurostat

In [ ]:
msw_data = pd.read_csv("../data_downloads/env_wasmun_linear.csv")
country_codes = pd.read_csv("../data_saved/country_codes.csv")

In [ ]:
#Changes for country codes are not needed since the updated file is saved
#nuts_0 = pd.read_csv("data_downloads/locationNUTS-0.csv")

In [ ]:
#List of EU27+4 regions that are under study
eu27 = country_codes["Code"].values

In [ ]:
units = msw_data["unit"].unique()
wst_oper = msw_data["wst_oper"].unique()
time_period = msw_data["TIME_PERIOD"].unique() 

__Breakdown of Waste Management Operations:__ <br> <br>
__RECYCLING:__ <br>
RCY = RCY_M + RCY_C_D + RCY_PRP_REU <br> <br>
__INCINERATION:__  <br>
DSP_I_RCV_E = RCV_E+ DSP_I <br> <br>
__ALL TREATMENT:__ <br>
TRT = DSP_I_RCV_E + DSP_L_OTH + RCY


# MSW Data Cleaning

In [ ]:
# Selecting data with E27 regions
eu_msw = msw_data.loc[(msw_data["geo"].isin(eu27))]
eu_msw.drop(["DATAFLOW","LAST UPDATE","freq","OBS_FLAG"],axis = 'columns', inplace = True)

In [ ]:
#Pivoting table to perform calculations
eu_msw = eu_msw.pivot(index=['geo','TIME_PERIOD','unit'], columns='wst_oper', values='OBS_VALUE')

In [ ]:
# Mapping country code to 3-letter codes
eu_msw = eu_msw.reset_index()
eu_msw["Country"] = eu_msw.merge(country_codes, left_on='geo', right_on = "Code", how = 'left')["NUTS_code"]
eu_msw.drop(['geo'], axis = 'columns', inplace = True)

## Removing NaNs 

In [ ]:
nan_count = eu_msw.isna().sum()

__Cleaning Prepare for Reuse columns__

In [ ]:
eu_msw.loc[eu_msw['PRP_REU'].isna(),'PRP_REU'] = 0

__Cleaning Energy recovery and Incineration data__
- If one of the values of DSP_I and RCV_E is NaN, then it is assumed to be zero and 
  DSP_I_RCV_E is a summation of DSP_I, RSV_E.
- If all three are zero, then if the volume for other treatment methods add up to
  volume of treatment, then everything is zero

In [ ]:
#Changing NaN values when one of DSP_I, RCV_E is NaN: 
eu_msw.loc[(eu_msw['DSP_I'].isna()) & (eu_msw['RCV_E'].notna()), 'DSP_I']=0
eu_msw.loc[(eu_msw['RCV_E'].isna()) & (eu_msw['DSP_I'].notna()), 'RCV_E']=0
eu_msw['DSP_I_RCV_E']=np.where((eu_msw['DSP_I_RCV_E'].isna()) & (eu_msw['DSP_I'].notna()) &
                            (eu_msw['RCV_E'].notna()), eu_msw[['DSP_I','RCV_E']].sum(axis=1),
                            eu_msw['DSP_I_RCV_E'])

When all three are NaN, Checking if the volumes of other treatment methods add up, and changing NaN

In [ ]:
eu_msw['DSP_I']=np.where((eu_msw['DSP_I_RCV_E'].isna()) & (eu_msw['RCY'].notna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['RCY']==eu_msw['TRT']),
                                       0, eu_msw['DSP_I'])
eu_msw['RCV_E']=np.where((eu_msw['DSP_I_RCV_E'].isna()) & (eu_msw['RCY'].notna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['RCY']==eu_msw['TRT']),
                                       0, eu_msw['RCV_E'])
eu_msw['DSP_I_RCV_E']=np.where((eu_msw['DSP_I_RCV_E'].isna()) & (eu_msw['RCY'].notna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['RCY']==eu_msw['TRT']),
                                       0, eu_msw['DSP_I_RCV_E'])

__Cleaning Recycling data__

In [ ]:
# Similar to incineration, changing NaN when one of the two recycling processes is NaN
eu_msw.loc[(eu_msw['RCY_C_D'].isna()) & (eu_msw['RCY_M'].notna()), 'RCY_C_D']=0
eu_msw.loc[(eu_msw['RCY_M'].isna()) & (eu_msw['RCY_C_D'].notna()), 'RCY_M']=0
eu_msw['RCY']=np.where((eu_msw['RCY'].isna()) & (eu_msw['RCY_C_D'].notna()) &
                            (eu_msw['RCY_M'].notna()), eu_msw[['RCY_C_D','RCY_M']].sum(axis=1),
                            eu_msw['RCY']) 

When all three are NaN, Checking if the volumes of other treatment methods add up, and changing NaN

In [ ]:
eu_msw['RCY_M']=np.where((eu_msw['RCY'].isna()) & (eu_msw['DSP_I_RCV_E'].notna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['DSP_I_RCV_E']==eu_msw['TRT']),
                                       0, eu_msw['RCY_M'])
eu_msw['RCY_C_D']=np.where((eu_msw['DSP_I_RCV_E'].notna()) & (eu_msw['RCY'].isna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['DSP_I_RCV_E']==eu_msw['TRT']),
                                       0, eu_msw['RCY_C_D'])
eu_msw['RCY']=np.where((eu_msw['DSP_I_RCV_E'].notna()) & (eu_msw['RCY'].isna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['DSP_I_RCV_E']==eu_msw['TRT']),
                                       0, eu_msw['RCY'])

## Missing data

### Check for missing data

In [ ]:
country_check = country_codes["NUTS_code"].iloc[:31]
missing_data = []
for country in country_check:
    for year in np.arange(2010, 2022):
        if eu_msw.loc[(eu_msw["Country"]==country) & (eu_msw["TIME_PERIOD"]==year)].empty:
            missing_data.append([country,year])
            
missing_data = pd.DataFrame(missing_data, columns = ["Country", "TIME_PERIOD"])

### OECD data comparison
Loading OECD MSW data and selecting relevant countries, treatment methods, variables, and units of measure

In [ ]:
oecd_msw = pd.read_csv("data_downloads/msw_oecd.csv")
oecd_units = oecd_msw["Unit of measure"].unique()
oecd_msw = oecd_msw.loc[oecd_msw["Unit of measure"].isin(oecd_units[1:3])]
oecd_msw = oecd_msw.loc[oecd_msw["REF_AREA"].isin(country_codes["NUTS_code"])]
measure = ['Total waste generated',
          'Material recovery (Recycling + Composting)', 'Composting',
          'Recycling', 'Other recovery',
          'Incineration without energy recovery', 'Memo: total incineration',
          'Landfill', 'Disposal operations',
          'Other disposal', 'Recovery operations',
          'Amounts designated for treatment operations',
          'Incineration with energy recovery']
oecd_msw = oecd_msw.loc[oecd_msw["Measure"].isin(measure)]
oecd_vars = ["REF_AREA","Reference area", "Measure","UNIT_MEASURE", "Unit of measure","TIME_PERIOD", "OBS_VALUE","UNIT_MULT"]
oecd_msw = oecd_msw[oecd_vars]

In [ ]:
oecd_msw_percap = oecd_msw.loc[oecd_msw["UNIT_MEASURE"]=='KG_PS'] 
oecd_msw_tonnes = oecd_msw.loc[oecd_msw["UNIT_MEASURE"]=='T'] 

In [ ]:
oecd_msw_tonnes = oecd_msw_tonnes.pivot(index=["REF_AREA","Reference area","UNIT_MEASURE", "Unit of measure","TIME_PERIOD","UNIT_MULT"], columns=['Measure'], values='OBS_VALUE')
oecd_msw_tonnes = oecd_msw_tonnes.reset_index()

In [ ]:
oecd_msw_percap = oecd_msw_percap.pivot(index=["REF_AREA","Reference area","UNIT_MEASURE", "Unit of measure","TIME_PERIOD","UNIT_MULT"], columns=['Measure'], values='OBS_VALUE')
oecd_msw_percap = oecd_msw_percap.reset_index()

### Getting Missing Data for UK from OECD data

In [ ]:
# Getting the values for UK
oecd_uk = oecd_msw_tonnes.loc[(oecd_msw_tonnes["REF_AREA"]=='GBR')&(oecd_msw_tonnes["TIME_PERIOD"].isin(time_period))]
oecd_uk_cap = oecd_msw_percap.loc[(oecd_msw_percap["REF_AREA"]=='GBR')&(oecd_msw_percap["TIME_PERIOD"].isin(time_period))]

In [ ]:
#columns to drop : 'Recovery operations', 'Other recovery','Disposal operations','Other disposal'
map_trt = {'Total waste generated':'GEN',
          'Material recovery (Recycling + Composting)':'RCY', 
          'Composting' : 'RCY_C_D',
          'Recycling': 'RCY_M',
          'Incineration without energy recovery': 'DSP_I', 
          'Memo: total incineration': 'DSP_I_RCV_E',
          'Landfill':'DSP_L_OTH', 
          'Amounts designated for treatment operations': 'TRT',
          'Incineration with energy recovery': 'RCV_E'}

In [ ]:
oecd_uk.drop(['Recovery operations', 'Other recovery','Disposal operations','Other disposal', "Unit of measure", "UNIT_MULT", "Reference area"], axis = 'columns', inplace = True)
oecd_uk = oecd_uk.rename(columns = map_trt)
oecd_uk = oecd_uk.rename(columns = {"REF_AREA":'Country',"UNIT_MEASURE":'unit'})
oecd_uk["unit"]='THS_T'
#Assume Prepare for reuse is 0
oecd_uk["PRP_REU"] = 0

In [ ]:
#Per capita data
oecd_uk_cap.drop([ "Unit of measure", "UNIT_MULT", "Reference area"], axis = 'columns', inplace = True)
oecd_uk_cap = oecd_uk_cap.rename(columns = map_trt)
oecd_uk_cap = oecd_uk_cap.rename(columns = {"REF_AREA":'Country',"UNIT_MEASURE":'unit'})
oecd_uk_cap["unit"]='KG_HAB'
#Assume Prepare for reuse is 0
oecd_uk_cap["PRP_REU"] = 0

Separating per cap data and total volume data


In [ ]:
eu_msw_cap = eu_msw.loc[eu_msw["unit"] == units[0]]
eu_msw = eu_msw.loc[eu_msw["unit"] == units[1]]

Filling in missing data by interpolation when the neighbouring years are available

In [ ]:
missing_data['unit'] = 'THS_T'

In [ ]:
eu_msw = pd.concat([eu_msw,missing_data.loc[(missing_data['TIME_PERIOD']<2020)&(missing_data['Country']!='GBR')]])
eu_msw.sort_values(by=['Country', 'TIME_PERIOD'], inplace = True, ignore_index = True)

In the missing data, the countries GRC (Greece) and HRV (Croatia) have some missing values that cannot be filled in by linear interpolation

For the rows with nan, which are not GRC or HRV (since they have some missing values), use interpolate to fill in data 


In [ ]:
excluded_countries = ['GRC', 'HRV']
interpolate_df = eu_msw[~eu_msw["Country"].isin(excluded_countries)].copy()
interpolate_df = interpolate_df.interpolate()
eu_msw.update(interpolate_df, overwrite = False)

Repeat the steps for per cap data

In [ ]:
country_check = country_codes["NUTS_code"].iloc[:31]
missing_data_cap = []
for country in country_check:
    for year in np.arange(2010, 2022):
        if eu_msw_cap.loc[(eu_msw_cap["Country"]==country) & (eu_msw_cap["TIME_PERIOD"]==year)].empty:
            missing_data_cap.append([country,year])
            
missing_data_cap = pd.DataFrame(missing_data_cap, columns = ["Country", "TIME_PERIOD"])

In [ ]:
eu_msw_cap = pd.concat([eu_msw_cap,missing_data_cap.loc[(missing_data_cap['TIME_PERIOD']<2020)&(missing_data_cap['Country']!='GBR')]])
eu_msw_cap.sort_values(by=['Country', 'TIME_PERIOD'], inplace = True, ignore_index = True)


<br>
for the rows with nan, which are not GRC or HRV (since they have some missing values), use interpolate to fill in data <br>


In [ ]:
excluded_countries = ['GRC', 'HRV']
interpolate_df = eu_msw_cap[~eu_msw_cap["Country"].isin(excluded_countries)].copy()
interpolate_df = interpolate_df.interpolate()
eu_msw_cap.update(interpolate_df, overwrite = False)

Adding missing values for UK from OECD data

In [ ]:
eu_msw = pd.concat([eu_msw[eu_msw["Country"]!='GBR'],oecd_uk], ignore_index = True)

In [ ]:
countries_to_group = country_codes["NUTS_code"][:-1].to_list()
df_filtered = eu_msw[eu_msw['Country'].isin(countries_to_group)]

Create a new row for 'EU 27+4' by summing values for specified countries

In [ ]:
df_filtered = df_filtered.groupby(['TIME_PERIOD', 'unit'], as_index=False).sum()
df_filtered['Country'] = 'EU27+4'
df_filtered = df_filtered[~df_filtered['TIME_PERIOD'].isin([2020,2021])]

In [ ]:
eu_msw = pd.concat([eu_msw, df_filtered], ignore_index=True)

In [ ]:
eu_msw.to_csv("data_saved/EU_MSW_Cleaned_Data.csv")

Adding missing values for UK from OECD data

In [ ]:
eu_msw_cap = pd.concat([eu_msw_cap[eu_msw_cap["Country"]!='GBR'],oecd_uk_cap], ignore_index = True)

In [ ]:
eu_msw_cap.to_csv("data_saved/EU_MSW_percap_Cleaned.csv", index = False)